# Project 2: Data Cleaning & Validation Tool

Hi! This is my second data analytics project — I'm a 1st year CSE student learning this stuff mostly by building small tools and figuring things out as I go.

In Project 1 I built an Excel report generator using a clean dataset (Superstore Sales). This time I wanted to practice something different: actually cleaning messy data, since that's apparently most of what real data work looks like.

**Dataset:** [Dirty Cafe Sales](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) from Kaggle -> 10,000 rows of fake cafe transactions that are purposely messed up (missing values, "ERROR"/"UNKNOWN" placeholder text, etc.) so people can practice cleaning on it.

**What I built:**
A tool that takes the messy CSV and:
1. Figures out what's actually wrong with it (this took longer than I expected , turns out "missing" isn't just blank cells, some of it was hidden as text like "UNKNOWN")
2. Cleans it up , fills in missing values, removes duplicate rows, fixes inconsistent text, parses dates properly
3. Spits out an Excel file with two sheets:
   - **Flagged** -> the original messy data with the problem cells colored red (missing) or yellow (duplicate), so you can see what was wrong
   - **Cleaned** -> the actual fixed version, ready to use

**New things I learned this project:** `dropna()` / `fillna()`, `drop_duplicates()`, `.str.strip()` / `.str.lower()`, `pd.to_datetime()`, and how to do conditional formatting (colored cells) in openpyxl.

**One honest note:** this dataset's Transaction IDs are all unique, so it didn't actually have any real duplicate rows in it. To practice `drop_duplicates()` properly I added 5 fake duplicates myself,that part is clearly marked below so it's not hiding anything.


## Step 1: Loading the data and taking a first look

First just loading the CSV and seeing what I'm working with.

In [ ]:
import pandas as pd
df=pd.read_csv("dirty_cafe_sales.csv")
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [ ]:
df.isnull().sum()

,0
Transaction ID,0
Item,333
Quantity,138
Price Per Unit,179
Total Spent,173
Payment Method,2579
Location,3265
Transaction Date,159


In [ ]:
df['Payment Method'].unique()

array(['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan],
      dtype=object)

In [ ]:
df['Location'].unique()

array(['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR'], dtype=object)

In [ ]:
df['Item'].unique()

array(['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'UNKNOWN',
       'Sandwich', nan, 'ERROR', 'Juice', 'Tea'], dtype=object)

## Step 2: Missing values

This is where it got interesting,pandas only catches real blank/NaN cells by default. But this dataset also has "UNKNOWN" and "ERROR" typed in as if they were real values, which pandas has no way of knowing are actually missing data unless I tell it. Had to find that out the hard way.

In [ ]:
df.replace(['UNKNOWN','ERROR'],pd.NA,inplace=True)

In [ ]:
df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'])
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'])
df['Total Spent'] = pd.to_numeric(df['Total Spent'])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  object 
 1   Item              9031 non-null   object 
 2   Quantity          9521 non-null   float64
 3   Price Per Unit    9467 non-null   float64
 4   Total Spent       9498 non-null   float64
 5   Payment Method    6822 non-null   object 
 6   Location          6039 non-null   object 
 7   Transaction Date  9540 non-null   object 
dtypes: float64(3), object(5)
memory usage: 625.1+ KB


In [ ]:
df['Transaction Date']=pd.to_datetime(df['Transaction Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9031 non-null   object        
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   object        
 6   Location          6039 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), object(4)
memory usage: 625.1+ KB


In [ ]:
print(df['Quantity'].median())
print(df['Price Per Unit'].median())
print(df['Total Spent'].median())

3.0
3.0
8.0


In [ ]:
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].median())
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())

df.isnull().sum()

,0
Transaction ID,0
Item,969
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,3178
Location,3961
Transaction Date,460


In [ ]:
df['Item'] = df['Item'].fillna('Unknown')
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')

df.isnull().sum()

,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,460


In [ ]:
df = df.dropna(subset=['Transaction Date'])

df.isnull().sum()

,0
Transaction ID,0
Item,0
Quantity,0
Price Per Unit,0
Total Spent,0
Payment Method,0
Location,0
Transaction Date,0


In [ ]:
print(len(df))

9540


## Step 3: Duplicates

Wanted to check for duplicate rows and remove them but as I found out, this dataset doesn't actually have any (see note below).

In [ ]:
print(df.duplicated().sum())

0


This dataset did not naturally contain duplicate rows (each transaction has a unique Transaction ID). To properly test and demonstrate the deduplication logic of this cleaning tool, a small number of duplicate rows were intentionally injected below.


In [ ]:
duplicate_rows=df.sample(5,random_state=42)
df=pd.concat([df,duplicate_rows],ignore_index=True)
df.duplicated().sum()

np.int64(5)

In [ ]:
print(df.duplicated().sum())
print(len(df))

5
9545


In [ ]:
df = df.drop_duplicates()

print(df.duplicated().sum())
print(len(df))

0
9540


## Step 4: Cleaning up text formatting

Checking for extra spaces and inconsistent capitalization in the text columns, and standardizing everything.

In [ ]:
df['Item'].apply(lambda x:x!=x.strip()).sum()

np.int64(0)

In [ ]:
print(df['Payment Method'].apply(lambda x: x != x.strip()).sum())
print(df['Location'].apply(lambda x: x != x.strip()).sum())

0
0


In [ ]:
print(df['Item'].apply(lambda x: x != x.lower() and x.lower() in df['Item'].str.lower().unique()).sum())

9540


In [ ]:
print(df['Item'].nunique())
print(df['Item'].str.lower().nunique())

9
9


In [ ]:
df['Item'] = df['Item'].str.strip().str.lower()
df['Payment Method'] = df['Payment Method'].str.strip().str.lower()
df['Location'] = df['Location'].str.strip().str.lower()

df['Item'].unique()

array(['coffee', 'cake', 'cookie', 'salad', 'smoothie', 'unknown',
       'sandwich', 'juice', 'tea'], dtype=object)

## Step 5: Checking the dates make sense

Just making sure the date column parsed correctly and the values are actually reasonable (not something like the year 1900 or 2099).

In [ ]:
print(df['Transaction Date'].min())
print(df['Transaction Date'].max())

2023-01-01 00:00:00
2023-12-31 00:00:00


## Step 6: Rebuilding the "before" data for the report

Ran into a design problem here -> by this point I'd already cleaned everything in `df`, which means all the "problems" were fixed or deleted, so there was nothing left to highlight for the report. Had to go back, reload the raw CSV separately, and rebuild a version that keeps the mess so I can actually show what was wrong. Lesson learned: next time I'd track problems *during* cleaning instead of after.

In [ ]:
df_original = pd.read_csv('dirty_cafe_sales.csv')
df_original.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [ ]:
print(df_original.loc[df_original['Transaction ID'] == 'TXN_4271903', 'Total Spent'])
print(df_original['Payment Method'].unique())

2    ERROR
Name: Total Spent, dtype: object
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]


In [ ]:
missing_map=df_original.isnull()|df_original.isin(['UNKNOWN','ERROR'])
missing_map.sum()

,0
Transaction ID,0
Item,969
Quantity,479
Price Per Unit,533
Total Spent,502
Payment Method,3178
Location,3961
Transaction Date,460


In [ ]:
print(len(df_original))

10000


In [ ]:
duplicate_ids = duplicate_rows['Transaction ID'].tolist()
print(duplicate_ids)

['TXN_8413949', 'TXN_4447856', 'TXN_6763296', 'TXN_2023802', 'TXN_5999810']


In [ ]:
df_flagged = df_original.copy()
df_flagged = pd.concat([df_flagged, duplicate_rows], ignore_index=True)

is_duplicate_row = df_flagged['Transaction ID'].isin(duplicate_ids)
print(is_duplicate_row.sum())
print(len(df_flagged))

10
10005


In [ ]:
print(df_flagged.loc[df_flagged['Transaction ID'] == 'TXN_4271903', 'Total Spent'])
print(df_flagged['Payment Method'].unique())

2    ERROR
Name: Total Spent, dtype: object
['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]


In [ ]:
missing_map_flagged = df_flagged.isnull() | df_flagged.isin(['UNKNOWN', 'ERROR'])
print(missing_map_flagged.sum())
print(len(missing_map_flagged))


Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64
10005


In [ ]:
print(len(df))
df.head()

9540


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,coffee,2.0,2.0,4.0,credit card,takeaway,2023-09-08
1,TXN_4977031,cake,4.0,3.0,12.0,cash,in-store,2023-05-16
2,TXN_4271903,cookie,4.0,1.0,8.0,credit card,in-store,2023-07-19
3,TXN_7034554,salad,2.0,5.0,10.0,unknown,unknown,2023-04-27
4,TXN_3160411,coffee,2.0,2.0,4.0,digital wallet,in-store,2023-06-11


## Step 7: Building the actual Excel file

Using openpyxl to write both sheets and color the problem cells. This part took a lot of trial and error to get the highlighting to line up with the right cells , worth it though, learned a lot about how easy it is for two DataFrames to quietly get out of sync.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill


In [ ]:
red_fill=PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
yellow_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")

In [ ]:
wb= Workbook()
ws_flagged=wb.active
ws_flagged.title="Flagged"
for col_num, column_title in enumerate(df_flagged.columns, start=1):
    ws_flagged.cell(row=1, column=col_num, value=column_title)

print("Header written")


Header written


In [ ]:
for row_num,row_data in enumerate(df_flagged.itertuples(index=False),start=2):
  for col_num,value in enumerate(row_data,start=1):
    cell=ws_flagged.cell(row=row_num,column=col_num,value=value)
    column_name=df_flagged.columns[col_num-1]


    if missing_map_flagged.iloc[row_num - 2, col_num - 1]:
            cell.fill = red_fill
    elif is_duplicate_row.iloc[row_num - 2]:
            cell.fill = yellow_fill

print("Data rows written")

Data rows written


In [ ]:
ws_cleaned=wb.create_sheet(title="Cleaned")

for col_num,column_title in enumerate(df.columns,start=1):
  ws_cleaned.cell(row=1,column=col_num,value=column_title)

for row_num, row_data in enumerate(df.itertuples(index=False), start=2):
    for col_num, value in enumerate(row_data, start=1):
        ws_cleaned.cell(row=row_num, column=col_num, value=value)

print("Cleaned sheet written")

Cleaned sheet written


In [ ]:
from openpyxl.utils import get_column_letter

for ws in [ws_flagged, ws_cleaned]:
    for col_num in range(1, len(df_flagged.columns) + 1):
        column_letter = get_column_letter(col_num)
        ws.column_dimensions[column_letter].width = 18

print("Columns widened")

Columns widened


In [ ]:
from openpyxl.styles import Font

header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")

for ws in [ws_flagged, ws_cleaned]:
    for col_num in range(1, len(df_flagged.columns) + 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill

print("Headers styled")

Headers styled


In [ ]:
legend_row = len(df_flagged) + 3  # a few rows below the last data row

ws_flagged.cell(row=legend_row, column=1, value="Legend:").font = Font(bold=True)

ws_flagged.cell(row=legend_row + 1, column=1, value="Missing value (filled/flagged)")
ws_flagged.cell(row=legend_row + 1, column=2).fill = red_fill

ws_flagged.cell(row=legend_row + 2, column=1, value="Duplicate row")
ws_flagged.cell(row=legend_row + 2, column=2).fill = yellow_fill

print("Legend added")

Legend added


In [ ]:
wb.save('cleaning_validation_report.xlsx')

from google.colab import files
files.download('cleaning_validation_report.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>